In [20]:
!pip install mlflow dagshub -q

In [21]:
## Setup
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor, Pool

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit

import mlflow
import mlflow.catboost

import dagshub

dagshub.init(repo_owner='ejoba22', repo_name='walmart-sales-forecasting', mlflow=True)

EXPERIMENT_NAME = "CatBoost_Training"
mlflow.set_experiment(EXPERIMENT_NAME)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

Initialized MLflow to track repo "ejoba22/walmart-sales-forecasting"

Repository ejoba22/walmart-sales-forecasting initialized!

In [22]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

models_to_unregister = [
    "walmart_xgboost",
    "DLinear_Walmart",
    "walmart_timesfm_pipeline",
]

for model_name in models_to_unregister:
    try:
        client.delete_registered_model(name=model_name)
        print(f"Deleted: {model_name}")
    except Exception as e:
        print(f"Could not delete {model_name}: {e}")

Deleted: walmart_xgboost
Deleted: DLinear_Walmart


## 1. Data Cleaning 

In [ ]:
BASE = '/kaggle/input/datasets/elenejobava/walmart-feature-engineering/results/'

train_clean = pd.read_parquet(BASE + 'train_clean.parquet')
test_clean  = pd.read_parquet(BASE + 'test_clean.parquet')

with mlflow.start_run(run_name="CatBoost_Cleaning"):
    n_null_train = train_clean.isnull().sum().sum()
    n_null_test  = test_clean.isnull().sum().sum()
    n_negative   = (train_clean['Weekly_Sales'] < 0).sum()

    mlflow.log_param("train_rows", len(train_clean))
    mlflow.log_param("test_rows", len(test_clean))
    mlflow.log_param("negative_sales_kept", True)
    mlflow.log_metric("null_cells_train", int(n_null_train))
    mlflow.log_metric("null_cells_test", int(n_null_test))
    mlflow.log_metric("negative_sales_rows", int(n_negative))

    print(f"train_clean: {train_clean.shape}, test_clean: {test_clean.shape}")
    print(f"Negative sales rows kept as legitimate returns: {n_negative}")
    print(f"Remaining null cells — train: {n_null_train}, test: {n_null_test}")

## 2. Feature Engineering

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

HOLIDAYS = {
    'SuperBowl':    pd.to_datetime(['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08']),
    'LaborDay':     pd.to_datetime(['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06']),
    'Thanksgiving': pd.to_datetime(['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29']),
    'Christmas':    pd.to_datetime(['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27']),
}

NAME_MAP = {'SuperBowl': 'super_bowl', 'LaborDay': 'labor_day',
            'Thanksgiving': 'thanksgiving', 'Christmas': 'christmas'}

LAGS = [1, 2, 3, 4, 5, 6, 12, 26, 52]
WINDOWS = [4, 8, 12, 26, 52]


class WalmartFeatureEngineer(BaseEstimator, TransformerMixin):
    """Same as the XGBoost notebook's version, except transform() keeps the raw 'Type'
    column (dropped there) so CatBoost can use it as a native categorical feature."""

    def fit(self, X, y=None):
        df = X.copy()
        if 'Weekly_Sales' not in df.columns and y is not None:
            df['Weekly_Sales'] = np.asarray(y)
        df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

        self.train_history_ = df[['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday']].copy()

        self.dept_avg_ = df.groupby('Dept')['Weekly_Sales'].mean()
        self.store_dept_avg_ = df.groupby(['Store', 'Dept'])['Weekly_Sales'].mean()
        self.store_avg_ = df.groupby('Store')['Weekly_Sales'].mean()

        holiday_sens = (
            df.groupby('Dept')
            .apply(lambda g: g.loc[g['IsHoliday'], 'Weekly_Sales'].mean() /
                              g.loc[~g['IsHoliday'], 'Weekly_Sales'].mean()
                              if (~g['IsHoliday']).any() and g.loc[~g['IsHoliday'], 'Weekly_Sales'].mean() else np.nan,
                   include_groups=False)
        )
        self.dept_holiday_sensitivity_ = holiday_sens

        cv = (
            df.groupby(['Store', 'Dept'])['Weekly_Sales']
            .apply(lambda x: x.std() / x.mean() if x.mean() else 0, include_groups=False)
        )
        self.store_dept_cv_ = cv

        dept_rank = (
            df.groupby(['Store', 'Dept'])['Weekly_Sales'].mean()
            .groupby(level='Store').rank(ascending=False)
        )
        self.dept_rank_in_store_ = dept_rank
        self.dept_avg_lt_1000_ = (self.dept_avg_ < 1000)

        df_cal = self._add_calendar(df)
        self.store_dept_week_avg_ = df_cal.groupby(['Store', 'Dept', 'Week'])['Weekly_Sales'].mean()
        return self

    def _add_calendar(self, df):
        df = df.copy()
        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['DayOfYear'] = df['Date'].dt.dayofyear
        df['Quarter'] = df['Date'].dt.quarter
        df['Year_norm'] = df['Year'] - 2010
        df['Week_sin'] = np.sin(2 * np.pi * df['Week'] / 52)
        df['Week_cos'] = np.cos(2 * np.pi * df['Week'] / 52)
        df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
        df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

        dates_arr = df['Date'].values.astype('datetime64[D]')
        for name, dates in HOLIDAYS.items():
            holiday_arr = dates.values.astype('datetime64[D]')
            diffs_days = (dates_arr[:, None] - holiday_arr[None, :]).astype('timedelta64[D]').astype(float)
            diffs_weeks = diffs_days / 7
            idx = np.argmin(np.abs(diffs_weeks), axis=1)
            nearest = diffs_weeks[np.arange(len(dates_arr)), idx]

            df[f'weeks_to_{name}'] = nearest
            df[f'near_{name}'] = (np.abs(nearest) <= 2).astype(int)
            df[f'before_{name}'] = (nearest < 0).astype(int)
            df[f'is_{NAME_MAP[name]}'] = df['Date'].isin(dates).astype(int)
        return df

    def transform(self, X):
        df = X.copy()
        has_target = 'Weekly_Sales' in df.columns
        if not has_target:
            df['Weekly_Sales'] = np.nan

        df['any_markdown'] = (df[MARKDOWN_COLS].fillna(0) > 0).any(axis=1).astype(int)
        df['Type_enc'] = df['Type'].map({'A': 2, 'B': 1, 'C': 0})
        df['Size_log'] = np.log1p(df['Size'])

        hist = self.train_history_[['Store', 'Dept', 'Date', 'Weekly_Sales']].copy()
        hist['_is_new'] = False
        new = df[['Store', 'Dept', 'Date', 'Weekly_Sales']].copy()
        new['_is_new'] = True
        combined = pd.concat([hist, new], ignore_index=True)
        combined = combined.sort_values(['Store', 'Dept', 'Date'], kind='mergesort')
        combined = combined.drop_duplicates(subset=['Store', 'Dept', 'Date'], keep='last')

        for lag in LAGS:
            combined[f'lag_{lag}'] = combined.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(lag)
        for w in WINDOWS:
            grp = combined.groupby(['Store', 'Dept'])['Weekly_Sales']
            combined[f'rolling_mean_{w}'] = grp.shift(1).transform(lambda x: x.rolling(w, min_periods=1).mean())
            combined[f'rolling_std_{w}'] = grp.shift(1).transform(lambda x: x.rolling(w, min_periods=1).std())
            combined[f'rolling_max_{w}'] = grp.shift(1).transform(lambda x: x.rolling(w, min_periods=1).max())
        combined['expanding_mean'] = combined.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(1).transform(
            lambda x: x.expanding().mean())

        lag_cols = [c for c in combined.columns if c.startswith(('lag_', 'rolling_', 'expanding_'))]
        combined_new = combined[combined['_is_new']][['Store', 'Dept', 'Date'] + lag_cols]

        df = df.merge(combined_new, on=['Store', 'Dept', 'Date'], how='left')
        df = self._add_calendar(df)

        df = df.join(self.dept_avg_.rename('dept_avg_sales'), on='Dept')
        df = df.join(self.store_dept_avg_.rename('store_dept_avg_sales'), on=['Store', 'Dept'])
        df = df.join(self.store_dept_week_avg_.rename('store_dept_week_avg'), on=['Store', 'Dept', 'Week'])
        df = df.join(self.dept_holiday_sensitivity_.rename('dept_holiday_sensitivity'), on='Dept')
        df = df.join(self.store_dept_cv_.rename('store_dept_cv'), on=['Store', 'Dept'])
        df = df.join(self.store_avg_.rename('store_avg_sales'), on='Store')
        df = df.join(self.dept_rank_in_store_.rename('dept_rank_in_store'), on=['Store', 'Dept'])

        df['store_dept_avg_sales'] = df['store_dept_avg_sales'].fillna(df['dept_avg_sales'])
        df['store_dept_week_avg'] = df['store_dept_week_avg'].fillna(df['store_dept_avg_sales'])
        df['store_dept_cv'] = df['store_dept_cv'].fillna(0)
        df['dept_rank_in_store'] = df['dept_rank_in_store'].fillna(self.dept_rank_in_store_.median())

        df['IsHoliday'] = df['IsHoliday'].astype(int)
        df['total_markdown'] = df[MARKDOWN_COLS].fillna(0).sum(axis=1)
        df['active_markdown_count'] = (df[MARKDOWN_COLS].fillna(0) > 0).sum(axis=1)
        df['is_sparse_dept'] = df['Dept'].map(self.dept_avg_lt_1000_).fillna(False).astype(int)
        df['markdown_x_holiday'] = df['total_markdown'] * df['IsHoliday']

        self.feature_cols_ = [c for c in df.columns
                               if c not in ['Weekly_Sales', 'Date', 'Store_Dept']]
        out = df[self.feature_cols_].copy()
        return out

In [ ]:
fe = WalmartFeatureEngineer()
fe.fit(train_clean)
X_full = fe.transform(train_clean)
print(X_full.dtypes[['Store', 'Dept', 'Type', 'Type_enc']])
print(X_full[['Store', 'Dept', 'Type']].head())

## 3. Feature Selection

In [ ]:
fe = WalmartFeatureEngineer()
fe.fit(train_clean)
X_full = fe.transform(train_clean)
y_full = train_clean.sort_values(['Store', 'Dept', 'Date'])['Weekly_Sales'].values

X_full = X_full.drop(columns=['Type_enc'])

cat_features = ['Store', 'Dept', 'Type']
X_full[cat_features] = X_full[cat_features].astype(str)

feature_cols = X_full.columns.tolist()
cat_feature_idx = [feature_cols.index(c) for c in cat_features]

quick_model = CatBoostRegressor(
    iterations=200, depth=6, random_state=RANDOM_STATE,
    verbose=False
)
quick_model.fit(X_full, y_full, cat_features=cat_feature_idx)

importances = pd.Series(quick_model.get_feature_importance(), index=feature_cols).sort_values(ascending=False)

with mlflow.start_run(run_name="CatBoost_Feature_Selection"):
    mlflow.log_param("n_features_total", len(feature_cols))
    mlflow.log_param("cat_features", cat_features)

    fig, ax = plt.subplots(figsize=(8, 10))
    importances.head(30).sort_values().plot.barh(ax=ax)
    ax.set_title("Top 30 CatBoost Feature Importances")
    plt.tight_layout()
    mlflow.log_figure(fig, "top30_importance.png")
    plt.show()

print(importances.head(20))

## 4. Custom Metric WMAE

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

## 5. Time-Based Cross-Validation

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

CAT_FEATURES = ['Store', 'Dept', 'Type']

class CatBoostCategoricalPrep(BaseEstimator, TransformerMixin):
    """Casts categorical columns to string and fills categorical NaN with a placeholder.
    Leaves numeric columns untouched (including their NaNs) — CatBoost handles those natively."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        df = X.copy()
        for c in CAT_FEATURES:
            df[c] = df[c].astype(str).replace('nan', '__missing__')
        return df


train_sorted_raw = train_clean.sort_values('Date').reset_index(drop=True)
tscv = TimeSeriesSplit(n_splits=5)
fold_scores = []

GPU_PARAMS = dict(task_type='GPU', devices='0')

with mlflow.start_run(run_name="CatBoost_CV"):
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(train_sorted_raw)):
        train_slice = train_sorted_raw.iloc[tr_idx]
        val_slice = train_sorted_raw.iloc[val_idx]

        fe_fold = WalmartFeatureEngineer()
        fe_fold.fit(train_slice)
        X_tr = fe_fold.transform(train_slice).drop(columns=['Type_enc'])
        X_val = fe_fold.transform(val_slice).drop(columns=['Type_enc'])

        prep = CatBoostCategoricalPrep()
        X_tr = prep.transform(X_tr)
        X_val = prep.transform(X_val)

        y_tr = train_slice['Weekly_Sales'].values
        y_val = val_slice['Weekly_Sales'].values
        hol_val = val_slice['IsHoliday'].values

        model = CatBoostRegressor(
            iterations=400, depth=7, learning_rate=0.05,
            subsample=0.8, bootstrap_type='Bernoulli', random_state=RANDOM_STATE,
            verbose=False, **GPU_PARAMS
        )
        model.fit(X_tr, y_tr, cat_features=CAT_FEATURES)
        preds = model.predict(X_val)
        score = wmae(y_val, preds, hol_val)
        fold_scores.append(score)

        with mlflow.start_run(run_name=f"CatBoost_CV_fold{fold}", nested=True):
            mlflow.log_param("fold", fold)
            mlflow.log_param("train_rows", len(tr_idx))
            mlflow.log_param("val_rows", len(val_idx))
            mlflow.log_metric("fold_wmae", score)

        print(f"Fold {fold}: train={len(tr_idx)} val={len(val_idx)} WMAE={score:.2f}")

    mlflow.log_metric("cv_wmae_mean", float(np.mean(fold_scores)))
    mlflow.log_metric("cv_wmae_std", float(np.std(fold_scores)))

print(f"\nMean CV WMAE: {np.mean(fold_scores):.2f} +/- {np.std(fold_scores):.2f}")

## 6. Overfitting / Underfitting Diagnostics


In [ ]:
train_idx, val_idx = list(tscv.split(train_sorted_raw))[-1]
train_slice = train_sorted_raw.iloc[train_idx]
val_slice = train_sorted_raw.iloc[val_idx]

diag_fe = WalmartFeatureEngineer()
diag_fe.fit(train_slice)
X_tr = diag_fe.transform(train_slice).drop(columns=['Type_enc'])
X_val = diag_fe.transform(val_slice).drop(columns=['Type_enc'])

prep = CatBoostCategoricalPrep()
X_tr = prep.transform(X_tr)
X_val = prep.transform(X_val)

y_tr = train_slice['Weekly_Sales'].values
y_val = val_slice['Weekly_Sales'].values
hol_tr = train_slice['IsHoliday'].values
hol_val = val_slice['IsHoliday'].values

GPU_PARAMS = dict(task_type='GPU', devices='0')

configs = {
    "underfit": dict(iterations=30, depth=2, learning_rate=0.3,
                      l2_leaf_reg=50, subsample=0.5, bootstrap_type='Bernoulli'),
    "overfit": dict(iterations=1500, depth=12, learning_rate=0.3,
                     l2_leaf_reg=0.1, subsample=1.0, bootstrap_type='Bernoulli'),
    "tuned": dict(iterations=400, depth=7, learning_rate=0.05,
                   subsample=0.8, bootstrap_type='Bernoulli'),
}

results = {}
with mlflow.start_run(run_name="CatBoost_Overfit_Underfit"):
    for name, params in configs.items():
        model = CatBoostRegressor(random_state=RANDOM_STATE, verbose=False, **params, **GPU_PARAMS)
        model.fit(X_tr, y_tr, cat_features=CAT_FEATURES)

        train_wmae = wmae(y_tr, model.predict(X_tr), hol_tr)
        val_wmae = wmae(y_val, model.predict(X_val), hol_val)
        gap = val_wmae - train_wmae
        results[name] = dict(train_wmae=train_wmae, val_wmae=val_wmae, gap=gap)

        with mlflow.start_run(run_name=f"CatBoost_Overfit_Underfit_{name}", nested=True):
            mlflow.log_params(params)
            mlflow.log_metric("train_wmae", train_wmae)
            mlflow.log_metric("val_wmae", val_wmae)
            mlflow.log_metric("train_val_gap", gap)

        print(f"{name:10s} train_WMAE={train_wmae:8.1f}  val_WMAE={val_wmae:8.1f}  gap={gap:8.1f}")

print("\nSummary:")
for name, r in results.items():
    print(f"{name:10s} train={r['train_wmae']:.1f}  val={r['val_wmae']:.1f}  gap={r['gap']:.1f}")

## 7. Hyperparameter Tuning

In [ ]:
!pip install optuna -q
import optuna
from optuna.samplers import TPESampler

N_TRIALS = 15  # each trial = full 5-fold CV, so keep this modest; raise later if you have time to spare

def objective(trial):
    params = dict(
        iterations=trial.suggest_int("iterations", 200, 600, step=100),
        depth=trial.suggest_int("depth", 4, 9),
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 15.0, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        bootstrap_type='Bernoulli',
        random_state=RANDOM_STATE,
        verbose=False,
        **GPU_PARAMS,
    )

    fold_scores = []
    for tr_idx, val_idx in tscv.split(train_sorted_raw):
        train_slice = train_sorted_raw.iloc[tr_idx]
        val_slice = train_sorted_raw.iloc[val_idx]

        fe_fold = WalmartFeatureEngineer()
        fe_fold.fit(train_slice)
        X_tr = fe_fold.transform(train_slice).drop(columns=['Type_enc'])
        X_val = fe_fold.transform(val_slice).drop(columns=['Type_enc'])

        prep = CatBoostCategoricalPrep()
        X_tr = prep.transform(X_tr)
        X_val = prep.transform(X_val)

        y_tr = train_slice['Weekly_Sales'].values
        y_val = val_slice['Weekly_Sales'].values
        hol_val = val_slice['IsHoliday'].values

        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr, cat_features=CAT_FEATURES,
                   early_stopping_rounds=50)  # cuts wasted iterations on weak trials
        preds = model.predict(X_val)
        fold_scores.append(wmae(y_val, preds, hol_val))

    mean_score = float(np.mean(fold_scores))

    with mlflow.start_run(run_name=f"CatBoost_Tuning_trial{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("cv_wmae_mean", mean_score)
        mlflow.log_metric("cv_wmae_std", float(np.std(fold_scores)))

    return mean_score

with mlflow.start_run(run_name="CatBoost_HyperparameterTuning"):
    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=N_TRIALS)

    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_cv_wmae", study.best_value)

print(f"\nBest CV WMAE: {study.best_value:.2f}")
print("Best params:", study.best_params)

In [ ]:
from sklearn.pipeline import Pipeline

class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, cols_to_drop):
        self.cols_to_drop = cols_to_drop
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X.drop(columns=self.cols_to_drop)

final_pipeline = Pipeline([
    ('feature_engineer', WalmartFeatureEngineer()),
    ('drop_type_enc', DropColumns(['Type_enc'])),
    ('cat_prep', CatBoostCategoricalPrep()),
    ('catboost', CatBoostRegressor(
        **study.best_params,
        bootstrap_type='Bernoulli',
        random_state=RANDOM_STATE,
        verbose=False,
        **GPU_PARAMS,
    )),
])

final_pipeline.fit(
    train_clean,
    train_clean.sort_values(['Store', 'Dept', 'Date'])['Weekly_Sales'].values,
    catboost__cat_features=CAT_FEATURES,
)

test_preds = final_pipeline.predict(test_clean)

# NEW LINE — clip negatives: 7x more frequent (2.09% vs 0.30% in train) and more
# extreme (min -9292 vs actual train min -4988.9) than observed reality — a sign
# of poor extrapolation on sparse store/dept combos. Affects ~2% of predictions.
test_preds = np.clip(test_preds, 0, None)

print(f"test_preds shape: {test_preds.shape}, any NaN: {np.isnan(test_preds).any()}, "
      f"min: {test_preds.min():.1f}, max: {test_preds.max():.1f}")

In [ ]:
import cloudpickle

with mlflow.start_run(run_name="CatBoost_Final_Pipeline"):
    mlflow.log_params(study.best_params)
    mlflow.log_param("bootstrap_type", "Bernoulli")
    mlflow.log_metric("cv_wmae_best", study.best_value)
    mlflow.log_param("trained_on", "full_train_clean")
    mlflow.log_param("negative_pred_clip", True)

    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        name="catboost_pipeline",
        input_example=train_clean.head(5),
        serialization_format="cloudpickle",  
    )

    submission_df = pd.DataFrame({
        "Id": test_clean["Store"].astype(str) + "_" + test_clean["Dept"].astype(str) + "_" + test_clean["Date"].astype(str),
        "Weekly_Sales": test_preds,
    })
    submission_path = "/kaggle/working/catboost_submission.csv"
    submission_df.to_csv(submission_path, index=False)
    mlflow.log_artifact(submission_path)

print("Pipeline logged. Compare cv_wmae_best against your team's other models before Model Registry registration.")

In [23]:
week130_cutoff = pd.date_range(train_clean['Date'].min(), periods=143, freq='7D')[130]
week143_end = pd.date_range(train_clean['Date'].min(), periods=143, freq='7D')[142]

fair_train = train_sorted_raw[train_sorted_raw['Date'] < week130_cutoff]
fair_val = train_sorted_raw[(train_sorted_raw['Date'] >= week130_cutoff) & (train_sorted_raw['Date'] <= week143_end)]

fair_fe = WalmartFeatureEngineer()
fair_fe.fit(fair_train)
X_fair_tr = fair_fe.transform(fair_train).drop(columns=['Type_enc'])
X_fair_val = fair_fe.transform(fair_val).drop(columns=['Type_enc'])

prep = CatBoostCategoricalPrep()
X_fair_tr = prep.transform(X_fair_tr)
X_fair_val = prep.transform(X_fair_val)

fair_model = CatBoostRegressor(**study.best_params, bootstrap_type='Bernoulli', random_state=RANDOM_STATE, verbose=False, **GPU_PARAMS)
fair_model.fit(X_fair_tr, fair_train['Weekly_Sales'].values, cat_features=CAT_FEATURES)
fair_preds = fair_model.predict(X_fair_val)
fair_wmae = wmae(fair_val['Weekly_Sales'].values, fair_preds, fair_val['IsHoliday'].values)

print(f"CatBoost — Fair Week130-143 WMAE: {fair_wmae:.2f}")

with mlflow.start_run(run_name="CatBoost_FairComparison_Week130_143"):
    mlflow.log_metric("fair_wmae_week130_143", fair_wmae)

CatBoost — Fair Week130-143 WMAE: 1453.40
🏃 View run CatBoost_FairComparison_Week130_143 at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/2/runs/c38687eb749e44d99f862b451e19778a
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/2
